# VITAL Q corrected verification latency study

This notebook replays the numerical analysis of the executed version 4.2 campaign with durable saves after every history. It does not rerun signature generation or replace raw timing observations. The supplied campaign ran through the Python command-line entry points. Saved notebook outputs below were produced by executing these cells sequentially in one Python namespace.

The public verifier, protocol, fitted models, all raw observations, selective proofs, and execution receipts are provided alongside the notebook. The private pre-evaluation snapshot is not public preregistration.

In [1]:
from pathlib import Path
import sys, json, hashlib, numpy as np
ROOT = Path.cwd()
if not (ROOT / "release").exists():
    raise RuntimeError("Open this notebook from the research-package directory")
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "release"))
from research_tools import verify_freeze, ledger, sha
from analysis_v4 import summaries
print(verify_freeze(ROOT / "release", ROOT / "audit/freeze.json"))
analysis = json.loads((ROOT / "results/analysis.json").read_text())
validation = json.loads((ROOT / "audit/final_validation.json").read_text())
print(json.dumps(analysis["collection"], indent=2))

{'unchanged': True, 'files': 25}
{
  "histories": 90,
  "observations": 1080,
  "matched_blocks": 30,
  "start_unix": 1790355592.1562946,
  "end_unix": 1790362940.3869216,
  "all_counts_and_bytes_match": true,
  "max_rss_bytes": 290967552,
  "release_unchanged": true,
  "raw_sha256": "e9e23695cf49554f8093fc2c9997af0fb4242ea286cad8ccdbcc5e71909cbdc6"
}


## Collection and frozen inputs
The model form, fitting objective, complete schedule and acceptance criterion were fixed before evaluation. The full development rows are included, including isolated-mode observations that were not used in fitting.

In [2]:
development = [json.loads(s) for s in (ROOT / "measured/development.jsonl").read_text().splitlines()]
evaluation = [json.loads(s) for s in (ROOT / "measured/evaluation.jsonl").read_text().splitlines()]
assert len(development) == 864 and len(evaluation) == 1080
assert all(r["status"] == "ok" and r["bytes_match"] and r["observed_counts_match"] for r in development + evaluation)
for file, expected in json.loads((ROOT / "release/training_input_hashes.json").read_text()).items():
    assert sha(ROOT / file) == expected
print("Development observations:", len(development))
print("Evaluation observations:", len(evaluation))
print("Successful histories:", validation["successful_histories"])
print("Selective proofs reverified:", validation["public_selective_proofs_reverified"])

Development observations: 864
Evaluation observations: 1080
Successful histories: 162
Selective proofs reverified: 486


## Recompute every reported prediction and interval
All six fits are first recomputed from their development rows without changing the frozen model files. The 10,000-draw bootstrap preserves the matched evaluation block across all arms, checkpoints and timing modes. Coefficients are frozen; these intervals omit development-fit uncertainty.

In [3]:
models = json.loads((ROOT / "release/models.json").read_text())
from model import MODELS, fit
training = [r for r in development if r["mode"] == "balanced"]
for name, columns in MODELS.items():
    assert fit(training, columns) == models[name], name
print("All six development fits reproduce the frozen coefficients and diagnostics exactly")
replayed = summaries(evaluation, models, bootstrap=True)
for mode in ("balanced", "isolated"):
    assert replayed[mode] == analysis[mode]
    print(mode, "all model cells and intervals reproduce exactly")

All six development fits reproduce the frozen coefficients and diagnostics exactly
balanced all model cells and intervals reproduce exactly
isolated all model cells and intervals reproduce exactly


In [4]:
for mode in ("balanced", "isolated"):
    print("\nMode:", mode)
    item = replayed[mode]["100000"]
    print("Model | passes | median absolute relative error %")
    for name, result in item["models"].items():
        print(f"{name} | {result['point_passes']}/18 | {100*result['median_absolute_relative_error']:.4f}")
    print("Primary-form support rule:", item["support_rule_passed"])
    print("Paired error difference:", item["paired_median_absolute_error_difference"])


Mode: balanced
Model | passes | median absolute relative error %
validation_affine | 18/18 | 1.6316
without_intercept | 13/18 | 6.9641
without_record_validation | 10/18 | 13.7854
bytes_only | 6/18 | 39.8592
affine_bytes | 8/18 | 19.8731
tokens_plus_bytes | 8/18 | 16.9320
micro_components | 5/18 | 41.9775
Primary-form support rule: True
Paired error difference: {'point': -0.38227628883556425, 'interval_95': [-0.3862972756224374, -0.3649622490113035], 'unit': 'relative error; multiply by 100 for percentage points; negative favors corrected model'}

Mode: isolated
Model | passes | median absolute relative error %
validation_affine | 18/18 | 2.5545
without_intercept | 13/18 | 6.7075
without_record_validation | 9/18 | 13.8112
bytes_only | 6/18 | 39.2288
affine_bytes | 8/18 | 20.6607
tokens_plus_bytes | 8/18 | 17.7107
micro_components | 5/18 | 41.7516
Primary-form support rule: True
Paired error difference: {'point': -0.3667431638792403, 'interval_95': [-0.37967094986728755, -0.357292632026

## Exact storage accounting
Payload bytes are excluded. The storage reductions are consequences of the selected batch envelope and do not constitute evidence of model accuracy.

In [5]:
sizes = ledger(100000, 6, 12)
print(json.dumps(sizes, indent=2))
assert sizes["archive_bytes"]["N"] == 1853710363
assert sizes["proof_bytes_at_supplied_sibling_count"]["N"] == 28900
assert sizes["proof_bytes_at_supplied_sibling_count"]["E"] == 29324
for arm in ("E", "D"):
    print(arm, "evidence reduction %", 100*(1-sizes["archive_bytes"][arm]/sizes["archive_bytes"]["N"]))

{
  "record_count": 100000,
  "generations": 6,
  "batches": 25,
  "classification": "analytical_not_measured",
  "archive_bytes": {
    "N": 1853710363,
    "E": 110847988,
    "D": 125825150
  },
  "proof_bytes_at_supplied_sibling_count": {
    "N": 28900,
    "E": 29324,
    "D": 245766
  },
  "sibling_count": 12,
  "full_token_checks": {
    "N": 600000,
    "E": 150,
    "D": 300
  }
}
E evidence reduction % 94.02020994150314
D evidence reduction % 93.21225405481536


## Balance and execution receipts
Balanced task positions and ordered predecessor pairs are verified against the raw rows. The receipt is a private hosted snapshot record, not a DOI or a trusted timestamp.

In [6]:
from collections import Counter
for arm in "NED":
    group = [r for r in evaluation if r["arm"] == arm and r["mode"] == "balanced"]
    positions = Counter((r["renewals"], r["query"], r["position"]) for r in group)
    pairs = Counter((tuple(r["predecessor"]), r["renewals"], r["query"]) for r in group if r["predecessor"] is not None)
    assert len(positions) == 36 and set(positions.values()) == {5}
    assert len(pairs) == 30 and set(pairs.values()) == {5}
    print(arm, "positions and predecessor pairs balanced")
freeze = json.loads((ROOT / "audit/freeze_receipt.json").read_text())
receipt = json.loads((ROOT / "audit/external_snapshot_receipt.json").read_text())
assert freeze["local_time_unix"] < receipt["receipt_recorded_unix"] < analysis["collection"]["start_unix"]
print("Freeze, hosted save receipt, evaluation ordering checked")
print("Public preregistration:", False)

N positions and predecessor pairs balanced
E positions and predecessor pairs balanced
D positions and predecessor pairs balanced
Freeze, hosted save receipt, evaluation ordering checked
Public preregistration: False


## Remaining scope
Full archives and full proofs were not retained; selective proofs and complete numerical logs were retained. Regenerating signatures will produce different archive hashes. The manuscript reports the actual support-rule outcome. Funding, conflicts, ethics determination for the secondary public-data example, author details, dated human-review scope, final AI disclosure and submission approval require factual author confirmation. The author has confirmed that a DOI is not required; public preregistration is not claimed.